# 04 · Tracking, labels, foundation-model embeddings, models

Put the foundation-model weights in `MyDrive/bmets/weights/`, using the file names in `configs/colab.yaml`.

In [ ]:
# --- setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['BMF_DRIVE'] = '/content/drive/MyDrive/bmets'                       # change if you use another folder
os.environ['BMF_DATA'] = os.environ['BMF_DRIVE'] + '/UCSD-BMETS-Longitudinal'   # Aspera download target (uploaded to Drive)
os.makedirs(os.environ['BMF_DRIVE'], exist_ok=True)
REPO = '/content/claude_projects'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rittik047/claude_projects.git $REPO
%cd /content/claude_projects/bmets-forecast
!pip -q install -e ".[dl,extra]"
CFG = 'configs/colab.yaml'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!bmf track --config $CFG
!bmf labels --config $CFG
!bmf features --config $CFG

In [ ]:
for enc in ['brainsegfounder', 'samed3d', 'random3d']:
    !bmf embed --config $CFG --encoder {enc} --device cuda

In [ ]:
for task in ['T2a', 'T2b']:
    for m in ['P0', 'P1', 'P2', 'P3', 'P5']:
        !bmf train --config $CFG --model {m} --task {task} --device cuda
    !bmf explain --config $CFG --task {task}
!bmf train --config $CFG --model P1 --task T3
!bmf train --config $CFG --model O1 --task T4
!bmf evaluate --config $CFG